In [ ]:
#1 Notebook description:

# this notebook is used to evaluate the market of assets and find potential assets to invest in
# with the best risk/reward characteristics. This notebook is intented to analyze a broad group of market assets
# and does NOT focus on any particular assets. Computing data for a large number of assets is computationally expensive and thus
# the analysis is relegated to other notebooks
# this notebook now also replaces the old 'Market Study - Statistical' notebook through the preset options below

In [ ]:
#2 Load libraries
import logging
logger = logging.getLogger('yfinance')
logger.disabled = True
logger.propagate = False
# Load libraries
import sys
import os
project_path = os.getcwd()
while not os.path.exists(os.path.join(project_path, "pyproject.toml")):
    parent = os.path.dirname(project_path)
    if parent == project_path:
        raise FileNotFoundError("Could not locate the project root from the current working directory.")
    project_path = parent
if project_path not in sys.path:
    sys.path.append(project_path)
from Quantapp.visualization import Plotter
from Quantapp.analytics import Metric, SeriesTransforms
from Quantapp.analytics.compute import latest
from Quantapp.analytics.series_utils import calculate_zscore
from Quantapp.data import MarketDataClient

import numpy as np
import json
import pandas as pd
from Quantapp.data import yf as qa_yf
from statsmodels.tsa.stattools import coint
from IPython.display import display
from plotly.subplots import make_subplots
from datetime import datetime
import statsmodels.api as sm
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import plotly.subplots as sp
import plotly.graph_objects as go
import pandas as pd
import holidays
import plotly.express as px
import concurrent.futures
from plotly.subplots import make_subplots
import plotly.graph_objects as go

#shut down warnings
import warnings
warnings.filterwarnings("ignore")
pio.templates.default = 'plotly_dark'



signal_metric = Metric()
series_transforms = SeriesTransforms()
qp = Plotter()
market_data = MarketDataClient()

def risk_adjusted_returns(data, windows, ratio_type='sharpe', risk_free_rate=0.0, annualization_factor=252):
    if isinstance(windows, (str, bytes)):
        raise ValueError("windows must be an integer or an iterable of integers")
    try:
        window_list = [int(window) for window in windows]
    except TypeError:
        window_list = [int(windows)]
    if not window_list or any(window <= 0 for window in window_list):
        raise ValueError("windows must contain positive integers")

    price_frame = data.to_frame(name=data.name or "price") if isinstance(data, pd.Series) else data
    if not isinstance(price_frame, pd.DataFrame):
        raise TypeError("data must be a pandas Series or DataFrame")

    returns = price_frame.pct_change()
    if isinstance(risk_free_rate, pd.Series):
        periodic_rate = risk_free_rate.astype(float).sort_index().reindex(returns.index).ffill()
    elif np.isscalar(risk_free_rate):
        periodic_rate = pd.Series((1.0 + float(risk_free_rate)) ** (1.0 / annualization_factor) - 1.0, index=returns.index)
    else:
        raise TypeError("risk_free_rate must be a scalar annual rate or a pandas Series")

    excess_returns = returns.sub(periodic_rate, axis=0)
    single_window = len(window_list) == 1
    single_series = price_frame.shape[1] == 1
    output = []
    for column in returns.columns:
        excess = excess_returns[column]
        for window in window_list:
            mean_excess = excess.rolling(window).mean()
            if ratio_type == 'sharpe':
                volatility = excess.rolling(window).std()
                ratio = np.sqrt(annualization_factor) * mean_excess / volatility
                ratio = ratio.where(volatility > 0)
            elif ratio_type == 'sortino':
                downside = excess.where(excess < 0, 0.0)
                downside_deviation = downside.rolling(window).apply(lambda values: np.sqrt((values**2).mean()), raw=True)
                ratio = np.sqrt(annualization_factor) * mean_excess / downside_deviation
            else:
                raise ValueError("Invalid ratio_type. Use 'sharpe' or 'sortino'.")

            ratio = ratio.replace([np.inf, -np.inf], np.nan)
            ratio.name = f"{ratio_type}_ratio_{window}" if single_window and single_series else f"{column}_{ratio_type}_{window}"
            output.append(ratio)
    return pd.concat(output, axis=1)


In [ ]:
#3 Parameters
time_frame_week = 7
time_frame_short = 21
time_frame_mid = 50
time_frame_long = 200
interval = '1d'
period = '10y'
risk_free_rate = 0.02 / 252 # Annualized risk-free rate divided by trading days
benchmark = 'SPY'
mode='standard'
analysis_preset = 'broad' # supported: 'broad', 'statistical'
preset_config = {
    'broad': {'asset_class': 'broad', 'sector': 'all'},
    'statistical': {'asset_class': 'equity', 'sector': 'financials'},
}
asset_class = preset_config[analysis_preset]['asset_class']
sector = preset_config[analysis_preset]['sector']
# Optional manual overrides:
# asset_class = 'equity'
# sector supported values: 'all', 'energy', 'materials', 'industrials', 'consumer_discretionary', 'consumer_staples', 'healthcare', 'financials', 'information_technology', 'communication_services', 'utilities', 'real_estate'
# sector = 'healthcare'

In [ ]:
#4 Broad market reference: cumulative returns, rolling Sharpe z-scores, and Sharpe spread z-scores
broad_market_ticker_map = {
    'Global Equities': 'ACWI',
    'U.S. Total Equity': 'VTI',
    'U.S. Large Cap': 'SPY',
    'International ex-U.S.': 'VXUS',
    'Emerging Markets': 'IEMG',
    'U.S. Aggregate Bonds': 'AGG',
    'Global Bonds': 'BNDW',
    'Broad Commodities': 'DBC',
    'Gold': 'GLD',
    'Crypto (Bitcoin)': 'BTC-USD',
    'U.S. Real Estate': 'VNQ',
    'U.S. Dollar Index': 'DX-Y.NYB'
}

# Match the default Risk Analysis history depth for block #12 comparisons.
broad_market_reference_period = '20y'

broad_market_reference = {
    name: qa_yf.Ticker(ticker).history(period=broad_market_reference_period, interval=interval)
    for name, ticker in broad_market_ticker_map.items()
}
broad_market_display_name_map = {
    name: f"{broad_market_ticker_map.get(name, 'N/A')} | {name}"
    for name in broad_market_ticker_map
}

def _normalize_market_index(data):
    normalized = data.copy()
    if not isinstance(normalized.index, pd.DatetimeIndex):
        return normalized
    index = normalized.index
    if index.tz is not None:
        index = index.tz_convert('UTC').tz_localize(None)
    normalized.index = index.normalize()
    return normalized.sort_index()

broad_market_reference = {
    name: _normalize_market_index(df)
    for name, df in broad_market_reference.items()
}

broad_market_close_map = {
    name: df['Close'].dropna().sort_index()
    for name, df in broad_market_reference.items()
    if not df.empty and 'Close' in df
}
broad_market_close = pd.DataFrame(broad_market_close_map).sort_index()

def _cumulative_return_series(series):
    valid = series.dropna()
    if valid.empty:
        return series
    return valid.div(valid.iloc[0]).sub(1).mul(100)

def plot_cumulative_return_overlay(price_data, title, default_label='10y', display_name_map=None):
    plot_data = price_data.dropna(how='all')
    if plot_data.empty or plot_data.shape[1] == 0:
        print(f'No data available for {title}.')
        return

    plot_columns = plot_data.columns.tolist()
    end_date = plot_data.index.max()
    min_date = plot_data.index.min()
    timeframe_offsets = [
        ('1y', pd.DateOffset(years=1)),
        ('2y', pd.DateOffset(years=2)),
        ('3y', pd.DateOffset(years=3)),
        ('5y', pd.DateOffset(years=5)),
        ('10y', pd.DateOffset(years=10)),
    ]

    cumulative_frames = {}
    for label, offset in timeframe_offsets:
        start_date = max(min_date, end_date - offset)
        window_prices = plot_data.loc[start_date:end_date]
        cumulative_frames[label] = window_prices.apply(_cumulative_return_series).reindex(window_prices.index)

    available_labels = [
        label for label, frame in cumulative_frames.items()
        if not frame.dropna(how='all').empty
    ]
    if not available_labels:
        print(f'No data available for {title}.')
        return

    if default_label not in available_labels:
        default_label = available_labels[0]

    default_frame = cumulative_frames[default_label]
    fig = go.Figure()
    for column in plot_columns:
        display_name = display_name_map.get(column, column) if display_name_map else column
        fig.add_trace(
            go.Scatter(
                x=default_frame.index,
                y=default_frame[column],
                mode='lines',
                name=display_name,
                showlegend=True
            )
        )

    buttons = []
    for label in available_labels:
        frame = cumulative_frames[label]
        x_values = [frame.index.tolist()] * len(plot_columns)
        y_values = [frame[column].tolist() for column in plot_columns]
        buttons.append(
            dict(
                label=label,
                method='update',
                args=[
                    {'x': x_values, 'y': y_values},
                    {'title': f'{title} ({label})'}
                ]
            )
        )

    fig.update_layout(
        title=f'{title} ({default_label})',
        height=700,
        template='plotly_dark',
        autosize=True,
        margin=dict(l=40, r=40, t=90, b=40),
        updatemenus=[
            dict(
                buttons=buttons,
                direction='down',
                showactive=True,
                active=available_labels.index(default_label),
                x=0.01,
                y=1.18,
                xanchor='left',
                yanchor='top'
            )
        ],
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0)
    )
    fig.update_xaxes(title_text='Date')
    fig.update_yaxes(title_text='Cumulative Return (%)')
    fig.add_hline(y=0, line_dash='dash', line_color='gray')
    return fig

def plot_reference_dataframe(data, title, yaxis_title, add_zero_line=False, sigma_lines=None, overlay_series=False, display_name_map=None, style=None):
    if data.empty:
        print(f'No data available for {title}.')
        return

    plot_data = data.dropna(how='all')
    if plot_data.empty or plot_data.shape[1] == 0:
        print(f'No data available for {title}.')
        return

    plot_columns = plot_data.columns.tolist()
    num_rows = 1 if overlay_series else len(plot_columns)
    end_date = plot_data.index.max()
    min_date = plot_data.index.min()
    zoom_ranges = [
        ('1y', pd.DateOffset(years=1)),
        ('2y', pd.DateOffset(years=2)),
        ('3y', pd.DateOffset(years=3)),
        ('5y', pd.DateOffset(years=5)),
        ('10y', pd.DateOffset(years=10)),
    ]
    default_label = '10y'
    default_start = max(min_date, end_date - dict(zoom_ranges)[default_label])

    def _xaxis_range_update(start_date, end_date):
        range_update = {}
        for axis_idx in range(1, num_rows + 1):
            axis_name = 'xaxis' if axis_idx == 1 else f'xaxis{axis_idx}'
            range_update[f'{axis_name}.range'] = [start_date, end_date]
        return range_update

    zoom_buttons = []
    for label, offset in zoom_ranges:
        start_date = max(min_date, end_date - offset)
        zoom_buttons.append(
            dict(
                label=label,
                method='relayout',
                args=[_xaxis_range_update(start_date, end_date)]
            )
        )

    subplot_spacing = 0.0 if overlay_series else min(0.03, 0.8 / max(num_rows, 2)) if num_rows > 1 else 0.0
    fig = make_subplots(
        rows=num_rows,
        cols=1,
        shared_xaxes=not overlay_series,
        vertical_spacing=subplot_spacing,
        subplot_titles=None if overlay_series else [display_name_map.get(column, column) if display_name_map else column for column in plot_columns]
    )
    for annotation in fig.layout.annotations:
        annotation.font = dict(size=10, color='rgba(220, 220, 220, 0.90)')

    for row_idx, column in enumerate(plot_columns, start=1):
        target_row = 1 if overlay_series else row_idx
        display_name = display_name_map.get(column, column) if display_name_map else column
        fig.add_trace(
            go.Scatter(
                x=plot_data.index,
                y=plot_data[column],
                mode='lines',
                name=display_name,
                showlegend=overlay_series
            ),
            row=target_row,
            col=1
        )

        if not overlay_series:
            if style:
                _apply_reference_panel_style(fig, row_idx, 1, style, add_zero_line=add_zero_line, sigma_lines=sigma_lines)
            else:
                if add_zero_line:
                    fig.add_hline(y=0, line_dash='dash', line_color='gray', row=row_idx, col=1)
                if sigma_lines:
                    for sigma in sigma_lines:
                        fig.add_hline(y=sigma, line_dash='dot', line_color='rgba(255,255,255,0.35)', row=row_idx, col=1)
                        fig.add_hline(y=-sigma, line_dash='dot', line_color='rgba(255,255,255,0.35)', row=row_idx, col=1)

        if not overlay_series:
            fig.update_yaxes(title_text=yaxis_title, row=row_idx, col=1)

    if overlay_series:
        if add_zero_line:
            fig.add_hline(y=0, line_dash='dash', line_color='gray', row=1, col=1)
        if sigma_lines:
            for sigma in sigma_lines:
                fig.add_hline(y=sigma, line_dash='dot', line_color='rgba(255,255,255,0.35)', row=1, col=1)
                fig.add_hline(y=-sigma, line_dash='dot', line_color='rgba(255,255,255,0.35)', row=1, col=1)
        fig.update_yaxes(title_text=yaxis_title, row=1, col=1)

    fig.update_layout(
        title=title,
        height=700 if overlay_series else max(420, 220 * num_rows + 120),
        template='plotly_dark',
        autosize=True,
        margin=dict(l=40, r=40, t=90, b=40),
        updatemenus=[
            dict(
                buttons=zoom_buttons,
                direction='down',
                showactive=True,
                active=[label for label, _ in zoom_ranges].index(default_label),
                x=0.01,
                y=1.18,
                xanchor='left',
                yanchor='top'
            )
        ]
    )
    if overlay_series:
        fig.update_layout(legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0))
    for row_idx in range(1, num_rows + 1):
        fig.update_xaxes(range=[default_start, end_date], row=row_idx, col=1)
    fig.update_xaxes(title_text='Date', row=num_rows, col=1)
    return fig

def _add_reference_zone(fig, row, col, y0, y1, fillcolor, opacity, label=None, font_color='rgba(235, 235, 235, 0.95)', x=0.5, font_size=14):
    fig.add_hrect(
        y0=y0,
        y1=y1,
        fillcolor=fillcolor,
        opacity=opacity,
        line_width=0,
        layer='below',
        row=row,
        col=col
    )
    if label:
        fig.add_annotation(
            x=x,
            y=(y0 + y1) / 2,
            xref='x domain',
            yref='y',
            text=label,
            showarrow=False,
            xanchor='center',
            yanchor='middle',
            font=dict(color=font_color, size=font_size),
            row=row,
            col=col
        )

def _add_std_level_annotations(fig, row, col, levels):
    labels = [(0, 'Mean')]
    for level in levels:
        labels.append((level, f'+{level} SD'))
        labels.append((-level, f'-{level} SD'))

    for y_value, label in labels:
        fig.add_annotation(
            x=0.985,
            y=y_value,
            xref='x domain',
            yref='y',
            text=label,
            showarrow=False,
            xanchor='left',
            yanchor='middle',
            font=dict(color='rgba(220, 220, 220, 0.90)', size=11),
            align='left',
            row=row,
            col=col
        )

def _apply_reference_panel_style(fig, row, col, style, add_zero_line=False, sigma_lines=None):
    if style == 'risk_detail':
        _add_reference_zone(fig, row, col, -1, 1, 'rgba(211, 211, 211, 0.18)', 1.0, label='Neutral')
        fig.add_annotation(
            x=0.5,
            y=-0.55,
            xref='x domain',
            yref='y',
            text='Bullish Neutral (on the way up)',
            showarrow=False,
            xanchor='center',
            yanchor='middle',
            font=dict(color='rgba(235, 235, 235, 0.90)', size=12),
            row=row,
            col=col
        )
        fig.add_annotation(
            x=0.5,
            y=0.55,
            xref='x domain',
            yref='y',
            text='Bearish Neutral (on the way down)',
            showarrow=False,
            xanchor='center',
            yanchor='middle',
            font=dict(color='rgba(235, 235, 235, 0.90)', size=12),
            row=row,
            col=col
        )
        _add_reference_zone(fig, row, col, -2, -1, 'rgba(0, 128, 0, 0.30)', 1.0, label='Accumulate', font_color='rgba(235, 255, 235, 0.95)')
        _add_reference_zone(fig, row, col, 1, 2, 'rgba(180, 0, 0, 0.30)', 1.0, label='Liquidate', font_color='rgba(255, 235, 235, 0.95)')
        levels = sigma_lines if sigma_lines is not None else [1, 2, 3]
        for sigma in levels:
            fig.add_hline(y=sigma, line_dash='dot', line_color='rgba(220, 220, 220, 0.55)', line_width=1, row=row, col=col)
            fig.add_hline(y=-sigma, line_dash='dot', line_color='rgba(220, 220, 220, 0.55)', line_width=1, row=row, col=col)
        if add_zero_line:
            fig.add_hline(y=0, line_dash='dash', line_color='rgba(160, 160, 160, 0.60)', line_width=1, row=row, col=col)

    elif style == 'risk_summary':
        _add_reference_zone(fig, row, col, -2, -1.5, 'rgba(180, 0, 0, 0.40)', 1.0, label='Liquidate', font_color='rgba(255, 235, 235, 0.95)')
        _add_reference_zone(fig, row, col, 1.5, 2, 'rgba(0, 128, 0, 0.55)', 1.0, label='Accumulate', font_color='rgba(235, 255, 235, 0.95)')
        fig.add_hline(y=0, line_color='rgba(255, 255, 255, 0.80)', line_width=1, row=row, col=col)
        levels = sigma_lines if sigma_lines is not None else [0.5, 1, 1.5, 2]
        for sigma in levels:
            fig.add_hline(y=sigma, line_dash='dot', line_color='rgba(220, 220, 220, 0.55)', line_width=1, row=row, col=col)
            fig.add_hline(y=-sigma, line_dash='dot', line_color='rgba(220, 220, 220, 0.55)', line_width=1, row=row, col=col)
        _add_std_level_annotations(fig, row, col, levels)

def plot_reference_dataframes_side_by_side(
    left_data,
    right_data,
    title,
    left_yaxis_title,
    right_yaxis_title,
    left_add_zero_line=False,
    right_add_zero_line=False,
    left_sigma_lines=None,
    right_sigma_lines=None,
    left_style=None,
    right_style=None,
    display_name_map=None
):
    left_plot_data = left_data.dropna(how='all') if left_data is not None else pd.DataFrame()
    right_plot_data = right_data.dropna(how='all') if right_data is not None else pd.DataFrame()

    left_is_empty = left_plot_data.empty or left_plot_data.shape[1] == 0
    right_is_empty = right_plot_data.empty or right_plot_data.shape[1] == 0
    if left_is_empty and right_is_empty:
        print(f'No data available for {title}.')
        return

    left_columns = [] if left_is_empty else left_plot_data.columns.tolist()
    right_columns = [] if right_is_empty else right_plot_data.columns.tolist()
    plot_columns = list(dict.fromkeys(left_columns + right_columns))
    num_rows = len(plot_columns)

    date_indexes = []
    if not left_is_empty:
        date_indexes.append(left_plot_data.index)
    if not right_is_empty:
        date_indexes.append(right_plot_data.index)

    end_date = max(index.max() for index in date_indexes)
    min_date = min(index.min() for index in date_indexes)
    zoom_ranges = [
        ('1y', pd.DateOffset(years=1)),
        ('2y', pd.DateOffset(years=2)),
        ('3y', pd.DateOffset(years=3)),
        ('5y', pd.DateOffset(years=5)),
        ('10y', pd.DateOffset(years=10)),
    ]
    default_label = '10y'
    default_start = max(min_date, end_date - dict(zoom_ranges)[default_label])

    def _xaxis_range_update(start_date, end_date):
        start_serialized = pd.Timestamp(start_date).isoformat()
        end_serialized = pd.Timestamp(end_date).isoformat()
        root_left_axis = 'xaxis' if num_rows == 1 else f'xaxis{((num_rows - 1) * 2) + 1}'
        root_right_axis = 'xaxis2' if num_rows == 1 else f'xaxis{((num_rows - 1) * 2) + 2}'
        return {
            f'{root_left_axis}.range[0]': start_serialized,
            f'{root_left_axis}.range[1]': end_serialized,
            f'{root_right_axis}.range[0]': start_serialized,
            f'{root_right_axis}.range[1]': end_serialized,
        }

    zoom_buttons = []
    for label, offset in zoom_ranges:
        start_date = max(min_date, end_date - offset)
        zoom_buttons.append(
            dict(
                label=label,
                method='relayout',
                args=[_xaxis_range_update(start_date, end_date)]
            )
        )

    asset_palette = px.colors.qualitative.Plotly + px.colors.qualitative.Dark24
    asset_colors = {
        column: asset_palette[idx % len(asset_palette)]
        for idx, column in enumerate(plot_columns)
    }

    subplot_titles = []
    for column in plot_columns:
        display_name = display_name_map.get(column, column) if display_name_map else column
        subplot_titles.append(f'{display_name} - Sharpe')
        subplot_titles.append(f'{display_name} - Spread')

    subplot_spacing = min(0.05, 1.0 / max(num_rows * 4, 8)) if num_rows > 1 else 0.0
    row_height = 300 if left_style or right_style else 240
    fig = make_subplots(
        rows=num_rows,
        cols=2,
        shared_xaxes='columns',
        vertical_spacing=subplot_spacing,
        horizontal_spacing=0.08,
        subplot_titles=subplot_titles
    )
    for annotation in fig.layout.annotations:
        annotation.font = dict(size=10, color='rgba(220, 220, 220, 0.90)')

    for row_idx, column in enumerate(plot_columns, start=1):
        display_name = display_name_map.get(column, column) if display_name_map else column
        if column in left_columns:
            fig.add_trace(
                go.Scatter(
                    x=left_plot_data.index,
                    y=left_plot_data[column],
                    mode='lines',
                    name=f'{display_name} - Sharpe',
                    line=dict(color=asset_colors[column], width=2),
                    showlegend=False
                ),
                row=row_idx,
                col=1
            )

            if left_style:
                _apply_reference_panel_style(fig, row_idx, 1, left_style, add_zero_line=left_add_zero_line, sigma_lines=left_sigma_lines)
            else:
                if left_add_zero_line:
                    fig.add_hline(y=0, line_dash='dash', line_color='gray', row=row_idx, col=1)
                if left_sigma_lines:
                    for sigma in left_sigma_lines:
                        fig.add_hline(y=sigma, line_dash='dot', line_color='rgba(255,255,255,0.35)', row=row_idx, col=1)
                        fig.add_hline(y=-sigma, line_dash='dot', line_color='rgba(255,255,255,0.35)', row=row_idx, col=1)
            fig.update_yaxes(title_text=left_yaxis_title, row=row_idx, col=1)

        if column in right_columns:
            fig.add_trace(
                go.Scatter(
                    x=right_plot_data.index,
                    y=right_plot_data[column],
                    mode='lines',
                    name=f'{display_name} - Spread',
                    line=dict(color=asset_colors[column], width=2),
                    showlegend=False
                ),
                row=row_idx,
                col=2
            )

            if right_style:
                _apply_reference_panel_style(fig, row_idx, 2, right_style, add_zero_line=right_add_zero_line, sigma_lines=right_sigma_lines)
            else:
                if right_add_zero_line:
                    fig.add_hline(y=0, line_dash='dash', line_color='gray', row=row_idx, col=2)
                if right_sigma_lines:
                    for sigma in right_sigma_lines:
                        fig.add_hline(y=sigma, line_dash='dot', line_color='rgba(255,255,255,0.35)', row=row_idx, col=2)
                        fig.add_hline(y=-sigma, line_dash='dot', line_color='rgba(255,255,255,0.35)', row=row_idx, col=2)
            fig.update_yaxes(title_text=right_yaxis_title, row=row_idx, col=2)

    fig.update_layout(
        title=title,
        height=max(620, row_height * num_rows + 180),
        template='plotly_dark',
        autosize=True,
        margin=dict(l=40, r=40, t=90, b=40),
        updatemenus=[
            dict(
                buttons=zoom_buttons,
                direction='down',
                showactive=True,
                active=[label for label, _ in zoom_ranges].index(default_label),
                x=0.01,
                y=1.12,
                xanchor='left',
                yanchor='top'
            )
        ]
    )

    fig.update_xaxes(range=[default_start, end_date], row=num_rows, col=1)
    fig.update_xaxes(range=[default_start, end_date], row=num_rows, col=2)
    fig.update_xaxes(title_text='Date', row=num_rows, col=1)
    fig.update_xaxes(title_text='Date', row=num_rows, col=2)
    return fig

def _display_stacked_plotly_figures(*figures):
    from IPython.display import display as ipy_display

    rendered_count = 0
    for fig in figures:
        if fig is None:
            continue
        fig.update_layout(autosize=True)
        ipy_display(fig)
        rendered_count += 1

    if rendered_count == 0:
        print('No broad market reference figures were generated.')

if broad_market_close.empty:
    print('No broad market reference data available.')
else:
    broad_market_close_plot = broad_market_close.ffill()

    broad_risk_free_daily_rate = pd.Series(risk_free_rate, index=broad_market_close.index, dtype=float)
    broad_risk_free_proxy = _normalize_market_index(qa_yf.Ticker('^IRX').history(period=broad_market_reference_period, interval=interval))
    if not broad_risk_free_proxy.empty and 'Close' in broad_risk_free_proxy:
        broad_risk_free_annual_yield = broad_risk_free_proxy['Close'].dropna().sort_index().div(100)
        if not broad_risk_free_annual_yield.empty:
            broad_risk_free_daily_rate = ((1 + broad_risk_free_annual_yield) ** (1 / 252) - 1).shift(1)
            broad_risk_free_daily_rate = broad_risk_free_daily_rate.reindex(broad_market_close.index).ffill()
            if broad_risk_free_daily_rate.dropna().empty:
                broad_risk_free_daily_rate = pd.Series(risk_free_rate, index=broad_market_close.index, dtype=float)

    broad_market_reference_benchmark = 'U.S. Large Cap'
    if broad_market_reference_benchmark not in broad_market_close_map:
        broad_market_reference_benchmark = next(iter(broad_market_close_map))

    benchmark_close_reference = broad_market_close_map[broad_market_reference_benchmark].dropna().sort_index()
    broad_market_rolling_sharpe_series = {}
    broad_market_sharpe_spread_series = {}
    for asset_name, asset_close in broad_market_close_map.items():
        analysis_index = asset_close.index.intersection(benchmark_close_reference.index).sort_values()
        if analysis_index.empty:
            continue

        asset_close_aligned = asset_close.reindex(analysis_index).dropna()
        benchmark_close_aligned = benchmark_close_reference.reindex(analysis_index).dropna()
        analysis_index = asset_close_aligned.index.intersection(benchmark_close_aligned.index).sort_values()
        if analysis_index.empty:
            continue

        asset_close_aligned = asset_close_aligned.reindex(analysis_index)
        benchmark_close_aligned = benchmark_close_aligned.reindex(analysis_index)
        aligned_risk_free_daily_rate = broad_risk_free_daily_rate.reindex(analysis_index).ffill()
        if aligned_risk_free_daily_rate.dropna().empty:
            aligned_risk_free_daily_rate = pd.Series(risk_free_rate, index=analysis_index, dtype=float)

        asset_sharpe = risk_adjusted_returns(
            asset_close_aligned,
            windows=[time_frame_long],
            ratio_type='sharpe',
            risk_free_rate=aligned_risk_free_daily_rate,
        ).iloc[:, 0]
        benchmark_sharpe = risk_adjusted_returns(
            benchmark_close_aligned,
            windows=[time_frame_long],
            ratio_type='sharpe',
            risk_free_rate=aligned_risk_free_daily_rate,
        ).iloc[:, 0].reindex(asset_sharpe.index)

        broad_market_rolling_sharpe_series[asset_name] = asset_sharpe
        broad_market_sharpe_spread_series[asset_name] = benchmark_sharpe - asset_sharpe

    broad_market_rolling_sharpe = pd.DataFrame(broad_market_rolling_sharpe_series).dropna(how='all')
    broad_market_rolling_sharpe_zscore = broad_market_rolling_sharpe.apply(calculate_zscore).dropna(how='all')

    broad_market_sharpe_spread = pd.DataFrame(broad_market_sharpe_spread_series).dropna(how='all')
    broad_market_sharpe_spread_zscore = broad_market_sharpe_spread.apply(calculate_zscore).dropna(how='all')
    broad_market_sharpe_spread_zscore = broad_market_sharpe_spread_zscore.drop(
        columns=[broad_market_reference_benchmark],
        errors='ignore'
    )

    broad_market_reference_benchmark_display = broad_market_display_name_map.get(
        broad_market_reference_benchmark,
        broad_market_reference_benchmark
    )
    broad_market_benchmark_sharpe_zscore = broad_market_rolling_sharpe_zscore[[broad_market_reference_benchmark]].dropna(how='all')
    broad_market_relative_sharpe_zscore = broad_market_rolling_sharpe_zscore.drop(
        columns=[broad_market_reference_benchmark],
        errors='ignore'
    )

    broad_market_cumulative_return_fig = plot_cumulative_return_overlay(
        broad_market_close_plot,
        'Broad Market Reference - Cumulative Returns',
        default_label='5y',
        display_name_map=broad_market_display_name_map
    )
    broad_market_large_cap_sharpe_fig = plot_reference_dataframe(
        broad_market_benchmark_sharpe_zscore,
        f'Broad Market Reference - {broad_market_reference_benchmark_display} Rolling Sharpe Z-Score',
        'Rolling Sharpe Z-Score',
        add_zero_line=True,
        sigma_lines=[1, 2, 3],
        display_name_map=broad_market_display_name_map,
        style='risk_detail'
    )
    broad_market_sharpe_spread_fig = plot_reference_dataframes_side_by_side(
        broad_market_relative_sharpe_zscore,
        broad_market_sharpe_spread_zscore,
        f'Broad Market Reference - Rolling Sharpe and Spread Z-Scores vs {broad_market_reference_benchmark_display}',
        'Rolling Sharpe Z-Score',
        'Sharpe Spread Z-Score',
        left_add_zero_line=True,
        right_add_zero_line=True,
        left_sigma_lines=[1, 2, 3],
        right_sigma_lines=[0.5, 1, 1.5, 2],
        left_style='risk_detail',
        right_style='risk_summary',
        display_name_map=broad_market_display_name_map
    )
    _display_stacked_plotly_figures(
        broad_market_cumulative_return_fig,
        broad_market_large_cap_sharpe_fig,
        broad_market_sharpe_spread_fig,
    )


In [ ]:
#5 Load: retrieve all tickers / prices
#------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
# Load: retrieve all tickers / prices / spreads for major markets
sp500 = qa_yf.download('SPY', period=period, interval=interval,progress=False)
risk_free_rate = qa_yf.download('^IRX', period=period, interval=interval, progress=False)

market_assets = market_data.get_market_assets()

if asset_class == 'broad':
    #Load: retrieve prices
    indices_df = market_data.generate_series(market_assets['INDICES'], columns=['Close'],period=period, interval=interval)
    sectors_df = market_data.generate_series(market_assets['SECTORS'], columns=['Close'],period=period, interval=interval)
    industries_df = market_data.generate_series(market_assets['INDUSTRIES'], columns=['Close'],period=period, interval=interval)
    bonds_df = market_data.generate_series(market_assets['BONDS'], columns=['Close'],period=period, interval=interval)
    precious_metals_df = market_data.generate_series(market_assets['PRECIOUS_METALS'], columns=['Close'],period=period, interval=interval)
    crypto_df = market_data.generate_series(market_assets['CRYPTO'], columns=['Close'],period=period, interval=interval)
    #crypto_df = crypto_df.loc[sp500.index]
    energy_df = market_data.generate_series(market_assets['ENERGY'], columns=['Close'],period=period, interval=interval)
    foreign_markets_df = market_data.generate_series(market_assets['FOREIGN_MARKETS'], columns=['Close'],period=period, interval=interval)
    primary_sector_etfs_df = market_data.generate_series(market_assets['PRIMARY_SECTORS'], columns=['Close'],period=period, interval=interval)
    capitalizations_df = market_data.generate_series(market_assets['CAPITALIZATIONS'], columns=['Close'],period=period, interval=interval)
    innovation_df = market_data.generate_series(market_assets['INNOVATION'], columns=['Close'],period=period, interval=interval)
    long_leveraged_df = market_data.generate_series(market_assets['LONG_LEVERAGE'], columns=['Close'],period=period, interval=interval)
    short_leveraged_df = market_data.generate_series(market_assets['SHORT_LEVERAGE'], columns=['Close'],period=period, interval=interval)
    single_factor_df = market_data.generate_series(market_assets['SINGLE_FACTOR'], columns=['Close'],period=period, interval=interval)
    multi_factor_df = market_data.generate_series(market_assets['MULTI_FACTOR'], columns=['Close'],period=period, interval=interval)
    minimum_volatility_df = market_data.generate_series(market_assets['MINIMUM_VOLATILITY'], columns=['Close'],period=period, interval=interval)


    etf_prices = pd.concat([
        indices_df,
        sectors_df,
        industries_df,
        bonds_df,
        precious_metals_df,
        #crypto_df,
        energy_df,
        foreign_markets_df,
        primary_sector_etfs_df,
        capitalizations_df,
        innovation_df,
        long_leveraged_df,
        short_leveraged_df,
        single_factor_df,
        multi_factor_df,
        minimum_volatility_df
    ], axis=1).loc[:, lambda df: ~df.columns.duplicated()]

    #test
    test_prices = pd.concat([indices_df,
                            sectors_df,
                            crypto_df
                            ], axis=1).loc[:, lambda df: ~df.columns.duplicated()]

    benchmark_series = etf_prices[benchmark]

    # List of dataframes for week and short time frames
    etf_dataframes = {
        "indices": indices_df,
        "sectors": sectors_df,
        "industries": industries_df,
        "bonds": bonds_df,
        "precious_metals": precious_metals_df,
        #"crypto": crypto_df,
        "energy": energy_df,
        "foreign_markets": foreign_markets_df,
        "primary_sector_etfs": primary_sector_etfs_df,
        "capitalizations": capitalizations_df,
        "innovation": innovation_df,
        "long_leveraged": long_leveraged_df,
        "short_leveraged": short_leveraged_df,
        "single_factor": single_factor_df,
        "multi_factor": multi_factor_df,
        "minimum_volatility": minimum_volatility_df
    }

    print('Computing the correlation matrix...')
    etf_dataframes_correlation_matrices = {key: value.corr() for key, value in etf_dataframes.items()}
elif asset_class == 'equity':
    xlk_holdings_df = market_data.generate_series(market_assets['XLK_HOLDINGS'], columns=['Close'], period=period, interval=interval)
    xlf_holdings_df = market_data.generate_series(market_assets['XLF_HOLDINGS'], columns=['Close'], period=period, interval=interval)
    xli_holdings_df = market_data.generate_series(market_assets['XLI_HOLDINGS'], columns=['Close'], period=period, interval=interval)
    xlb_holdings_df = market_data.generate_series(market_assets['XLB_HOLDINGS'], columns=['Close'], period=period, interval=interval)
    xlv_holdings_df = market_data.generate_series(market_assets['XLV_HOLDINGS'], columns=['Close'], period=period, interval=interval)
    xlu_holdings_df = market_data.generate_series(market_assets['XLU_HOLDINGS'], columns=['Close'], period=period, interval=interval)
    xly_holdings_df = market_data.generate_series(market_assets['XLY_HOLDINGS'], columns=['Close'], period=period, interval=interval)
    xlc_holdings_df = market_data.generate_series(market_assets['XLC_HOLDINGS'], columns=['Close'], period=period, interval=interval)
    xlp_holdings_df = market_data.generate_series(market_assets['XLP_HOLDINGS'], columns=['Close'], period=period, interval=interval)
    xle_holdings_df = market_data.generate_series(market_assets['XLE_HOLDINGS'], columns=['Close'], period=period, interval=interval)
    xlre_holdings_df = market_data.generate_series(market_assets['XLRE_HOLDINGS'], columns=['Close'], period=period, interval=interval)
    indices_df = market_data.generate_series(market_assets['INDICES'], columns=['Close'], period=period, interval=interval)
    etf_prices = pd.concat([indices_df], axis=1).loc[:, lambda df: ~df.columns.duplicated()]
    benchmark_series = etf_prices[benchmark]
    etf_dataframes = {
        "indices": indices_df,
        "xlk_holdings": xlk_holdings_df,
        "xlf_holdings": xlf_holdings_df,
        "xli_holdings": xli_holdings_df,
        "xlb_holdings": xlb_holdings_df,
        "xlv_holdings": xlv_holdings_df,
        "xlu_holdings": xlu_holdings_df,
        "xly_holdings": xly_holdings_df,
        "xlc_holdings": xlc_holdings_df,
        "xlp_holdings": xlp_holdings_df,
        "xle_holdings": xle_holdings_df,
        "xlre_holdings": xlre_holdings_df
    }
    # Mapping of sectors to their respective ETF holdings dataframes
    sector_mapping = {
        'energy': ('xle_holdings', xle_holdings_df),
        'materials': ('xlb_holdings', xlb_holdings_df),
        'industrials': ('xli_holdings', xli_holdings_df),
        'consumer_discretionary': ('xly_holdings', xly_holdings_df),
        'consumer_staples': ('xlp_holdings', xlp_holdings_df),
        'healthcare': ('xlv_holdings', xlv_holdings_df),
        'financials': ('xlf_holdings', xlf_holdings_df),
        'information_technology': ('xlk_holdings', xlk_holdings_df),
        'communication_services': ('xlc_holdings', xlc_holdings_df),
        'utilities': ('xlu_holdings', xlu_holdings_df),
        'real_estate': ('xlre_holdings', xlre_holdings_df)
    }

    if sector == 'all':
        # Include all sectors by updating the etf_dataframes dictionary
        etf_dataframes.update({key: df for key, df in sector_mapping.values()})
    elif sector in sector_mapping:
        # Include only the specific sector
        key, df = sector_mapping[sector]
        etf_dataframes = {
            "indices": indices_df,
            key: df
        }
        etf_prices = pd.concat([indices_df, df], axis=1).loc[:, lambda df: ~df.columns.duplicated()]
    else:
        supported_values = "'all', " + ", ".join([f"'{s}'" for s in sector_mapping.keys()])
        raise ValueError(f"Unsupported sector. Supported values are {supported_values}.")
    etf_dataframes_correlation_matrices = {key: value.corr() for key, value in etf_dataframes.items()}

else:
    raise ValueError("Unsupported asset class. Supported values are 'broad' and 'equity'.")


agg_dataframe = pd.concat(list(etf_dataframes.values()), axis=1)
agg_dataframe = agg_dataframe.loc[:,~agg_dataframe.columns.duplicated()]

In [ ]:
#6 Computations...

#1. return spreads between benchmark and all assets
#2. the sortino ratios for all assets
#3. the spreads between the sortino ratios of all assets and the benchmark


#compute the weekly, short, mid, and long term returns for the benchmark
sp500_monthly_returns = series_transforms.returns(sp500,frequency='monthly')
sp500_weekly_returns = series_transforms.returns(sp500,frequency='weekly')
sp500_daily_returns = series_transforms.returns(sp500,frequency='daily')

def benchmark_minus_asset_spread(asset_series, benchmark_series, time_frame, mode='standard', risk_free_rate=0.0):
    time_frame = int(time_frame)
    if time_frame <= 0:
        raise ValueError('time_frame must be a positive integer.')

    asset_data = asset_series.sort_index()
    if isinstance(benchmark_series, pd.Series):
        benchmark = benchmark_series
    elif isinstance(benchmark_series, pd.DataFrame):
        benchmark = benchmark_series['Close'] if 'Close' in benchmark_series.columns else benchmark_series.iloc[:, 0]
    else:
        benchmark = pd.Series(benchmark_series)
    benchmark = benchmark.dropna().sort_index()

    if mode == 'standard':
        asset_metric = asset_data.pct_change(time_frame)
        benchmark_metric = benchmark.pct_change(time_frame)
    elif mode == 'sortino':
        asset_metric = risk_adjusted_returns(
            asset_data,
            windows=[time_frame],
            ratio_type='sortino',
            risk_free_rate=risk_free_rate,
        )
        if isinstance(asset_data, pd.DataFrame) and asset_metric.shape[1] == asset_data.shape[1]:
            asset_metric = asset_metric.set_axis(asset_data.columns, axis=1)
        benchmark_metric = risk_adjusted_returns(
            benchmark,
            windows=[time_frame],
            ratio_type='sortino',
            risk_free_rate=risk_free_rate,
        ).iloc[:, 0]
    else:
        raise ValueError("Invalid mode. Use 'standard' or 'sortino'.")

    if isinstance(asset_metric, pd.Series):
        asset_metric = asset_metric.to_frame(name=asset_metric.name or 'asset')

    spread_df = pd.DataFrame(index=asset_metric.index)
    benchmark_metric = benchmark_metric.reindex(asset_metric.index)
    for col in asset_metric.columns:
        spread_df[f'Benchmark_minus_{col}'] = benchmark_metric - asset_metric[col]

    return spread_df

#the spreads between the benchmark and all assets
#-----------------------------------------------------------
#the spreads between the benchmark and all assets
#Create spreads for weekly time frame
print("Computing spreads between benchmark and all assets...")
benchmark_minus_etf_week = benchmark_minus_asset_spread(etf_prices, benchmark_series, time_frame=time_frame_week, mode=mode)

#Create spreads for 3 day time frame
print("Computing spreads between benchmark and all assets for the 3 day time frame...")
benchmark_minus_etf_3 = benchmark_minus_asset_spread(etf_prices, benchmark_series, time_frame=3, mode=mode)

#Create spreads for 9 day time frame
print("Computing spreads between benchmark and all assets for the 9 day time frame...")
benchmark_minus_etf_9 = benchmark_minus_asset_spread(etf_prices, benchmark_series, time_frame=9, mode=mode)

#display(benchmark_minus_etf_week)
#Create spreads for short time frame
print(f"Computing spreads between benchmark and all assets for the short time frame ({time_frame_short} days)...")
benchmark_minus_etf_short = benchmark_minus_asset_spread(etf_prices, benchmark_series, time_frame=time_frame_short, mode=mode)

#Create spreads for mid time frame
print(f"Computing spreads between benchmark and all assets for the mid time frame ({time_frame_mid} days)...")
benchmark_minus_etf_mid = benchmark_minus_asset_spread(etf_prices, benchmark_series, time_frame=time_frame_mid, mode=mode)

#create spreads for long time frame
print(f"Computing spreads between benchmark and all assets for the long time frame ({time_frame_long} days)...")
benchmark_minus_etf_long = benchmark_minus_asset_spread(etf_prices, benchmark_series, time_frame=time_frame_long, mode=mode)

#Create spreads for 400 day time frame
print("Computing spreads between benchmark and all assets for the 400 day time frame...")
benchmark_minus_etf_400 = benchmark_minus_asset_spread(etf_prices, benchmark_series, time_frame=400, mode=mode)

print(" ")
print("-----------------------------------------------------------------------")

#-----------------------------------------------------------

#the sortino ratios for all assets
print("computing the rolling sortino ratios for all assets for the 3 day time frame...")
rolling_sortino_ratios_etf_3 = risk_adjusted_returns(etf_prices, windows=[3], ratio_type='sortino').set_axis(etf_prices.columns, axis=1)

print("computing the rolling sortino ratios for all assets for the 9 day time frame...")
rolling_sortino_ratios_etf_9 = risk_adjusted_returns(etf_prices, windows=[9], ratio_type='sortino').set_axis(etf_prices.columns, axis=1)

print(f"computing the rolling sortino ratios for all assets for the short time frame ({time_frame_short} days)...")
rolling_sortino_ratios_etf_21 = risk_adjusted_returns(etf_prices, windows=[21], ratio_type='sortino').set_axis(etf_prices.columns, axis=1)

print(f"computing the rolling sortino ratios for all assets for the mid time frame ({time_frame_mid} days)...")
rolling_sortino_ratios_etf_50 = risk_adjusted_returns(etf_prices, windows=[50], ratio_type='sortino').set_axis(etf_prices.columns, axis=1)

print(f"computing the rolling sortino ratios for all assets for the long time frame ({time_frame_long} days)...")
rolling_sortino_ratios_etf_200 = risk_adjusted_returns(etf_prices, windows=[200], ratio_type='sortino').set_axis(etf_prices.columns, axis=1)

print("computing the rolling sortino ratios for all assets for the 400 day time frame...")
rolling_sortino_ratios_etf_400 = risk_adjusted_returns(etf_prices, windows=[400], ratio_type='sortino').set_axis(etf_prices.columns, axis=1)

print(" ")
print("-----------------------------------------------------------------------")

#the spreads between the sortino ratios of all assets and the benchmark
print("computing the rolling sortino ratios for all assets minus the benchmark for the 3 day time frame...")
rolling_sortino_ratios_benchmark_minus_etf_3 = -rolling_sortino_ratios_etf_3.sub(rolling_sortino_ratios_etf_3['SPY'], axis=0)

print("computing the rolling sortino ratios for all assets minus the benchmark for the 9 day time frame...")
rolling_sortino_ratios_benchmark_minus_etf_9 = -rolling_sortino_ratios_etf_9.sub(rolling_sortino_ratios_etf_9['SPY'], axis=0)

print(f"computing the rolling sortino ratios for all assets minus the benchmark for the short time frame ({time_frame_short} days)...")
rolling_sortino_ratios_benchmark_minus_etf_21 = -rolling_sortino_ratios_etf_21.sub(rolling_sortino_ratios_etf_21['SPY'], axis=0)

print(f"computing the rolling sortino ratios for all assets minus the benchmark for the mid time frame ({time_frame_mid} days)...")
rolling_sortino_ratios_benchmark_minus_etf_50 = -rolling_sortino_ratios_etf_50.sub(rolling_sortino_ratios_etf_50['SPY'], axis=0)

print(f"computing the rolling sortino ratios for all assets minus the benchmark for the long time frame ({time_frame_long} days)...")
rolling_sortino_ratios_benchmark_minus_etf_200 = -rolling_sortino_ratios_etf_200.sub(rolling_sortino_ratios_etf_200['SPY'], axis=0)

print("computing the rolling sortino ratios for all assets minus the benchmark for the 400 day time frame...")
rolling_sortino_ratios_benchmark_minus_etf_400 = -rolling_sortino_ratios_etf_400.sub(rolling_sortino_ratios_etf_400['SPY'], axis=0)

print(f"computing 10 year correlation metrics to {benchmark}...")
etf_daily_returns = etf_prices.pct_change(fill_method=None)
correlation_end = etf_daily_returns.index.max()
correlation_start = correlation_end - pd.DateOffset(years=10)
correlation_window = etf_daily_returns.loc[correlation_start:correlation_end]
correlation_tolerance = pd.Timedelta(days=45)
minimum_downside_observations = 30
correlation_metrics_10y = {
    'Pearson': {},
    'Spearman': {},
    'Kendall': {},
    'Downside Pearson': {}
}

for ticker in correlation_window.columns:
    aligned_returns = pd.concat([
        correlation_window[benchmark],
        correlation_window[ticker]
    ], axis=1).dropna()

    has_full_lookback = (
        not aligned_returns.empty
        and aligned_returns.index.min() <= correlation_start + correlation_tolerance
    )

    if has_full_lookback:
        benchmark_returns = aligned_returns.iloc[:, 0]
        asset_returns = aligned_returns.iloc[:, 1]
        downside_mask = benchmark_returns < 0
        downside_returns = aligned_returns.loc[downside_mask]

        correlation_metrics_10y['Pearson'][ticker] = benchmark_returns.corr(asset_returns, method='pearson')
        correlation_metrics_10y['Spearman'][ticker] = benchmark_returns.corr(asset_returns, method='spearman')
        correlation_metrics_10y['Kendall'][ticker] = benchmark_returns.corr(asset_returns, method='kendall')
        correlation_metrics_10y['Downside Pearson'][ticker] = (
            downside_returns.iloc[:, 0].corr(downside_returns.iloc[:, 1], method='pearson')
            if len(downside_returns) >= minimum_downside_observations else np.nan
        )
    else:
        for metric_name in correlation_metrics_10y:
            correlation_metrics_10y[metric_name][ticker] = np.nan

correlation_metrics_10y = {
    metric_name: pd.Series(metric_values)
    for metric_name, metric_values in correlation_metrics_10y.items()
}
correlation_10y = correlation_metrics_10y['Pearson']
'''
print(" ")
print("-----------------------------------------------------------------------")




'''

In [ ]:
#7 Correlation Comparison by Metric
correlation_palette = {
    'Q1 (Lowest)': '#ef4444',
    'Q2': '#f59e0b',
    'Q3': '#60a5fa',
    'Q4 (Highest)': '#22c55e'
}
correlation_plot_frames = {}
pearson_correlation_series = correlation_metrics_10y.get('Pearson', pd.Series(dtype=float))

for metric_name, metric_series in correlation_metrics_10y.items():
    metric_column = f'10 year {metric_name} Correlation to {benchmark}'
    metric_plot_df = metric_series.rename(metric_column).dropna().rename_axis('Ticker').reset_index()
    if metric_plot_df.empty:
        continue

    pearson_rank_reference = pearson_correlation_series.reindex(metric_plot_df['Ticker']).dropna().sort_values()
    pearson_rank_map = pd.Series(
        np.arange(1, len(pearson_rank_reference) + 1),
        index=pearson_rank_reference.index
    )

    quantile_count = min(4, len(metric_plot_df))
    quantile_labels = ['Q1 (Lowest)', 'Q2', 'Q3', 'Q4 (Highest)'][:quantile_count]
    metric_plot_df['Quantile'] = pd.qcut(
        metric_plot_df[metric_column].rank(method='first'),
        q=quantile_count,
        labels=quantile_labels
    )
    metric_plot_df = metric_plot_df.sort_values(by=metric_column, ascending=True).reset_index(drop=True)
    metric_plot_df['Position'] = np.arange(len(metric_plot_df))
    metric_plot_df['Current Rank'] = np.arange(1, len(metric_plot_df) + 1)
    metric_plot_df['Pearson Rank'] = metric_plot_df['Ticker'].map(pearson_rank_map)
    metric_plot_df['Rank Shift vs Pearson'] = metric_plot_df['Pearson Rank'] - metric_plot_df['Current Rank']
    metric_plot_df['Metric'] = metric_name
    metric_plot_df['Current Rank Label'] = metric_plot_df['Current Rank'].astype(int).astype(str)
    metric_plot_df['Pearson Rank Label'] = metric_plot_df['Pearson Rank'].apply(lambda value: 'N/A' if pd.isna(value) else str(int(value)))
    metric_plot_df['Rank Shift Label'] = metric_plot_df['Rank Shift vs Pearson'].apply(
        lambda value: 'No change'
        if pd.isna(value) or value == 0
        else f'Up {int(abs(value))} positions' if value > 0
        else f'Down {int(abs(value))} positions'
    )
    metric_plot_df['Color'] = metric_plot_df['Quantile'].map(correlation_palette)
    metric_plot_df['Shift Color'] = metric_plot_df['Rank Shift vs Pearson'].apply(
        lambda value: '#22c55e' if value > 0 else '#ef4444' if value < 0 else '#9ca3af'
    )
    correlation_plot_frames[metric_name] = (metric_column, metric_plot_df)

if correlation_plot_frames:
    default_metric = 'Pearson' if 'Pearson' in correlation_plot_frames else next(iter(correlation_plot_frames))
    default_column, default_plot_df = correlation_plot_frames[default_metric]
    default_positions = default_plot_df['Position'].tolist()
    default_ticker_order = default_plot_df['Ticker'].tolist()
    default_average_correlation = default_plot_df[default_column].mean()

    correlation_fig = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.08,
        subplot_titles=('Selected Correlation Metric', 'Rank Shift vs Pearson Order')
    )
    correlation_fig.add_trace(
        go.Bar(
            x=default_plot_df['Position'],
            y=default_plot_df[default_column],
            marker_color=default_plot_df['Color'],
            customdata=np.array(default_plot_df[['Ticker', 'Metric', 'Quantile', 'Current Rank Label', 'Pearson Rank Label', 'Rank Shift Label']]),
            hovertemplate='Ticker: %{customdata[0]}<br>Metric: %{customdata[1]}<br>Value: %{y:.2f}<br>Quantile: %{customdata[2]}<br>Current Rank: %{customdata[3]}<br>Pearson Rank: %{customdata[4]}<br>Shift vs Pearson: %{customdata[5]}<extra></extra>'
        ),
        row=1,
        col=1
    )
    correlation_fig.add_trace(
        go.Scatter(
            x=default_plot_df['Position'],
            y=np.full(len(default_plot_df), default_average_correlation, dtype=float),
            mode='lines',
            name='Average Correlation',
            line=dict(color='rgba(255, 255, 255, 0.90)', width=2, dash='dash'),
            hovertemplate='Average Correlation: %{y:.2f}<extra></extra>',
            showlegend=True
        ),
        row=1,
        col=1
    )
    correlation_fig.add_trace(
        go.Bar(
            x=default_plot_df['Position'],
            y=default_plot_df['Rank Shift vs Pearson'],
            marker_color=default_plot_df['Shift Color'],
            customdata=np.array(default_plot_df[['Ticker', 'Current Rank Label', 'Pearson Rank Label', 'Rank Shift Label']]),
            hovertemplate='Ticker: %{customdata[0]}<br>Rank Shift: %{y:+.0f}<br>Current Rank: %{customdata[1]}<br>Pearson Rank: %{customdata[2]}<br>Interpretation: %{customdata[3]}<extra></extra>'
        ),
        row=2,
        col=1
    )
    correlation_fig.add_hline(y=0, line_dash='dash', line_color='gray', row=2, col=1)

    metric_buttons = []
    for metric_name, (metric_column, metric_plot_df) in correlation_plot_frames.items():
        metric_average_correlation = metric_plot_df[metric_column].mean()
        metric_buttons.append(
            dict(
                label=metric_name,
                method='update',
                args=[
                    {
                        'x': [metric_plot_df['Position'], metric_plot_df['Position'], metric_plot_df['Position']],
                        'y': [
                            metric_plot_df[metric_column],
                            np.full(len(metric_plot_df), metric_average_correlation, dtype=float),
                            metric_plot_df['Rank Shift vs Pearson']
                        ],
                        'marker.color': [metric_plot_df['Color'], None, metric_plot_df['Shift Color']],
                        'customdata': [
                            np.array(metric_plot_df[['Ticker', 'Metric', 'Quantile', 'Current Rank Label', 'Pearson Rank Label', 'Rank Shift Label']]),
                            None,
                            np.array(metric_plot_df[['Ticker', 'Current Rank Label', 'Pearson Rank Label', 'Rank Shift Label']])
                        ]
                    },
                    {
                        'title': f'10 Year {metric_name} Correlation to {benchmark} by Quantile',
                        'xaxis.tickmode': 'array',
                        'xaxis.tickvals': metric_plot_df['Position'].tolist(),
                        'xaxis.ticktext': metric_plot_df['Ticker'].tolist(),
                        'xaxis2.tickmode': 'array',
                        'xaxis2.tickvals': metric_plot_df['Position'].tolist(),
                        'xaxis2.ticktext': metric_plot_df['Ticker'].tolist(),
                        'yaxis.title.text': 'Correlation',
                        'yaxis2.title.text': 'Rank Shift'
                    }
                ]
            )
        )

    correlation_fig.update_layout(
        title=f'10 Year {default_metric} Correlation to {benchmark} by Quantile',
        template='plotly_dark',
        autosize=True,
        height=760,
        margin=dict(l=40, r=40, t=120, b=120),
        updatemenus=[
            dict(
                buttons=metric_buttons,
                direction='down',
                showactive=True,
                x=0.01,
                y=1.22,
                xanchor='left',
                yanchor='top'
            )
        ],
        annotations=[
            dict(
                text='Quantiles: Q1 lowest, Q4 highest',
                xref='paper',
                yref='paper',
                x=1,
                y=1.22,
                showarrow=False,
                xanchor='right',
                yanchor='top',
                font=dict(size=11, color='white')
            )
        ]
    )
    correlation_fig.update_xaxes(title='', tickmode='array', tickvals=default_positions, ticktext=default_ticker_order, showticklabels=False, row=1, col=1)
    correlation_fig.update_xaxes(title='', tickmode='array', tickvals=default_positions, ticktext=default_ticker_order, tickangle=45, row=2, col=1)
    correlation_fig.update_yaxes(title='Correlation', row=1, col=1)
    correlation_fig.update_yaxes(title='Rank Shift', row=2, col=1)
    correlation_fig.show(config={'responsive': True})
else:
    print(f'No valid 10 year correlation data available for {benchmark}.')


In [ ]:
#7A Average Rolling Correlation Over Time
rolling_correlation_windows = {
    '21 Day Average Rolling Correlation': 21,
    '50 Day Average Rolling Correlation': 50,
    '200 Day Average Rolling Correlation': 200
}

rolling_correlation_universe = correlation_10y.dropna().index.tolist() if isinstance(correlation_10y, pd.Series) else []
if benchmark in rolling_correlation_universe and len(rolling_correlation_universe) > 1:
    rolling_correlation_universe = [ticker for ticker in rolling_correlation_universe if ticker != benchmark]
rolling_correlation_universe = [
    ticker for ticker in rolling_correlation_universe
    if ticker in etf_daily_returns.columns and ticker != benchmark
]

if benchmark not in etf_daily_returns.columns:
    print(f'Benchmark {benchmark} is not available in the daily return frame.')
elif not rolling_correlation_universe:
    print(f'No valid correlation universe available to compute average rolling correlation to {benchmark}.')
else:
    benchmark_daily_returns = etf_daily_returns[benchmark]
    rolling_average_correlation_map = {}

    for label, window in rolling_correlation_windows.items():
        rolling_correlation_frame = etf_daily_returns[rolling_correlation_universe].apply(
            lambda asset_returns: asset_returns.rolling(window).corr(benchmark_daily_returns)
        )
        rolling_average_correlation_map[label] = rolling_correlation_frame.mean(axis=1, skipna=True).dropna()

    non_empty_rolling_average_map = {
        label: series
        for label, series in rolling_average_correlation_map.items()
        if not series.empty
    }

    if not non_empty_rolling_average_map:
        print(f'No rolling average correlation data available for {benchmark}.')
    else:
        rolling_correlation_colors = {
            '21 Day Average Rolling Correlation': '#38bdf8',
            '50 Day Average Rolling Correlation': '#f59e0b',
            '200 Day Average Rolling Correlation': '#22c55e'
        }

        rolling_average_correlation_fig = make_subplots(
            rows=len(non_empty_rolling_average_map),
            cols=1,
            shared_xaxes=True,
            vertical_spacing=0.06,
            subplot_titles=list(non_empty_rolling_average_map.keys())
        )

        for annotation in rolling_average_correlation_fig.layout.annotations:
            annotation.font = dict(size=11, color='rgba(220, 220, 220, 0.90)')

        for row_idx, (label, correlation_series) in enumerate(non_empty_rolling_average_map.items(), start=1):
            correlation_mean = correlation_series.mean()
            correlation_min = min(correlation_series.min(), 0, correlation_mean)
            correlation_max = max(correlation_series.max(), 0, correlation_mean)
            correlation_span = correlation_max - correlation_min
            correlation_padding = max(correlation_span * 0.08, 0.02) if pd.notna(correlation_span) else 0.02
            rolling_average_correlation_fig.add_trace(
                go.Scatter(
                    x=correlation_series.index,
                    y=correlation_series,
                    mode='lines',
                    name=label,
                    line=dict(color=rolling_correlation_colors.get(label, '#60a5fa'), width=2),
                    hovertemplate='Date: %{x|%Y-%m-%d}<br>Average Correlation: %{y:.2f}<extra></extra>',
                    showlegend=False
                ),
                row=row_idx,
                col=1
            )
            rolling_average_correlation_fig.add_hline(
                y=correlation_mean,
                line_dash='dash',
                line_color='rgba(235, 235, 235, 0.75)',
                row=row_idx,
                col=1
            )
            rolling_average_correlation_fig.add_annotation(
                x=0.985,
                y=correlation_mean,
                xref='x domain',
                yref='y',
                text=f'Mean {correlation_mean:.2f}',
                showarrow=False,
                xanchor='right',
                yanchor='bottom',
                font=dict(size=10, color='rgba(235, 235, 235, 0.90)'),
                row=row_idx,
                col=1
            )
            rolling_average_correlation_fig.add_hline(
                y=0,
                line_dash='dot',
                line_color='rgba(180, 180, 180, 0.65)',
                row=row_idx,
                col=1
            )
            rolling_average_correlation_fig.add_annotation(
                x=0.985,
                y=0,
                xref='x domain',
                yref='y',
                text='Zero',
                showarrow=False,
                xanchor='right',
                yanchor='top',
                font=dict(size=10, color='rgba(200, 200, 200, 0.85)'),
                row=row_idx,
                col=1
            )
            rolling_average_correlation_fig.update_yaxes(
                title_text='Avg Corr',
                range=[correlation_min - correlation_padding, correlation_max + correlation_padding],
                row=row_idx,
                col=1
            )

        rolling_average_correlation_fig.update_layout(
            title=f'Average Rolling Correlation to {benchmark} Over Time',
            template='plotly_dark',
            autosize=True,
            height=max(780, 260 * len(non_empty_rolling_average_map) + 120),
            margin=dict(l=50, r=40, t=90, b=40)
        )
        rolling_average_correlation_fig.update_xaxes(title_text='Date', row=len(non_empty_rolling_average_map), col=1)
        rolling_average_correlation_fig.show(config={'responsive': True})


In [ ]:
#8 Create DataFrames for each time frame

sortino_ratio_by_window = {
    3: rolling_sortino_ratios_etf_3,
    9: rolling_sortino_ratios_etf_9,
    21: rolling_sortino_ratios_etf_21,
    50: rolling_sortino_ratios_etf_50,
    200: rolling_sortino_ratios_etf_200,
    400: rolling_sortino_ratios_etf_400,
}
benchmark_minus_sortino_by_window = {
    3: rolling_sortino_ratios_benchmark_minus_etf_3,
    9: rolling_sortino_ratios_benchmark_minus_etf_9,
    21: rolling_sortino_ratios_benchmark_minus_etf_21,
    50: rolling_sortino_ratios_benchmark_minus_etf_50,
    200: rolling_sortino_ratios_benchmark_minus_etf_200,
    400: rolling_sortino_ratios_benchmark_minus_etf_400,
}
compounding_metric_windows = [21, 50, 200]
correlation_column = f'10 year Correlation to {benchmark}'

def rolling_metric_mad_score(series):
    clean = pd.Series(series).replace([np.inf, -np.inf], np.nan).dropna().sort_index()
    if clean.empty:
        return pd.Series(dtype=float)

    median = clean.median()
    mad = (clean - median).abs().median()
    if mad == 0 or pd.isna(mad):
        return pd.Series(0.0, index=clean.index)

    return ((clean - median) / (1.4826 * mad)).dropna()

def calculate_rolling_compounding_metrics(price_data, window):
    price_frame = price_data.to_frame(name=price_data.name or 'price') if isinstance(price_data, pd.Series) else price_data.copy()
    returns = price_frame.pct_change(fill_method=None)
    arithmetic_mean = returns.rolling(window, min_periods=window).mean()
    geometric_mean = np.expm1(
        np.log1p(returns).rolling(window, min_periods=window).mean()
    )

    compounding_efficiency = (
        (1.0 + geometric_mean)
        .div(1.0 + arithmetic_mean)
        .replace([np.inf, -np.inf], np.nan)
    )
    volatility_drag = (arithmetic_mean - geometric_mean).replace([np.inf, -np.inf], np.nan)
    return compounding_efficiency, volatility_drag

def latest_mad_score_by_column(frame):
    def latest_score(column):
        mad_scores = rolling_metric_mad_score(column)
        if mad_scores.empty:
            return np.nan
        return mad_scores.iloc[-1]

    return frame.apply(latest_score)

compounding_efficiency_mad_by_window = {}
volatility_drag_mad_by_window = {}
for window in compounding_metric_windows:
    compounding_efficiency, volatility_drag = calculate_rolling_compounding_metrics(
        etf_prices,
        window=window,
    )
    compounding_efficiency_mad_by_window[window] = latest_mad_score_by_column(compounding_efficiency)
    volatility_drag_mad_by_window[window] = latest_mad_score_by_column(volatility_drag)

z_score_frames = {}
for window in sortino_ratio_by_window:
    z_score_frame = pd.DataFrame()
    z_score_frame[f'{window} day Sortino Ratio (z score)'] = latest(
        sortino_ratio_by_window[window],
        metric=signal_metric.z_score,
        dropna=False,
    )
    z_score_frame[f'{window} day Benchmark Minus ETF Sortino Ratio (z score)'] = latest(
        benchmark_minus_sortino_by_window[window],
        metric=signal_metric.z_score,
        dropna=False,
    )

    if window in compounding_metric_windows:
        empty_metric = pd.Series(index=etf_prices.columns, dtype=float)
        z_score_frame[f'{window} day Compounding Efficiency MAD Score'] = (
            compounding_efficiency_mad_by_window.get(window, empty_metric)
            .reindex(etf_prices.columns)
        )
        z_score_frame[f'{window} day Volatility Drag MAD Score'] = (
            volatility_drag_mad_by_window.get(window, empty_metric)
            .reindex(etf_prices.columns)
        )

    if window == 200:
        z_score_frame[correlation_column] = correlation_10y

    z_score_frame.sort_values(by=f'{window} day Sortino Ratio (z score)', ascending=True, inplace=True)
    z_score_frames[window] = z_score_frame.round(2)

z_score_3 = z_score_frames[3]
z_score_9 = z_score_frames[9]
z_score_21 = z_score_frames[21]
z_score_50 = z_score_frames[50]
z_score_200 = z_score_frames[200]
z_score_400 = z_score_frames[400]

# Combine the longer-horizon columns into the primary table
z_score_combined = pd.concat([
    z_score_21,
    z_score_50,
    z_score_200.drop(columns=[correlation_column], errors='ignore'),
    z_score_400,
], axis=1)

# Build a complementary short-horizon / correlation table
z_score_complementary = pd.concat([
    z_score_21,
    z_score_9,
    z_score_3,
    z_score_200[[correlation_column]],
], axis=1)

z_score_complementary = z_score_complementary[[
    '21 day Sortino Ratio (z score)',
    '21 day Benchmark Minus ETF Sortino Ratio (z score)',
    '9 day Sortino Ratio (z score)',
    '9 day Benchmark Minus ETF Sortino Ratio (z score)',
    '3 day Sortino Ratio (z score)',
    '3 day Benchmark Minus ETF Sortino Ratio (z score)',
    correlation_column,
]]

# Plot the combined metrics
combined_metrics_fig = qp.plot_z_score_combined(z_score_combined)
combined_metrics_fig.update_layout(
    title=(
        'Combined Sortino Z-Scores + '
        'Compounding Efficiency / Volatility Drag MAD Scores '
        '(21 / 50 / 200 / 400 Day)'
    ),
    autosize=True,
)
combined_metrics_fig.show(config={'responsive': True})

complementary_metrics_fig = qp.plot_z_score_combined(z_score_complementary)
complementary_metrics_fig.update_layout(
    title=f'Combined Z-Scores for Sortino Ratios (21 / 9 / 3 Day + 10 Year Correlation to {benchmark})',
    autosize=True,
)
complementary_metrics_fig.show(config={'responsive': True})

# Build an empirical threshold breach-rate table across short holding horizons.
from Quantapp.analytics.series_utils import calculate_historical_var_metrics

empirical_risk_lookback = 50
empirical_risk_confidence = 0.95
empirical_risk_alpha = 1.0 - empirical_risk_confidence
empirical_risk_horizons = list(range(0, 8))
empirical_risk_horizon_columns = [f'{horizon}D' for horizon in empirical_risk_horizons]
empirical_risk_open_prices = globals().get('empirical_risk_open_prices')
if not isinstance(empirical_risk_open_prices, pd.DataFrame):
    try:
        empirical_risk_open_prices = market_data.generate_series(
            etf_prices.columns.tolist(),
            columns=['Open'],
            period=period,
            interval=interval,
        )
    except Exception:
        empirical_risk_open_prices = pd.DataFrame(index=etf_prices.index, columns=etf_prices.columns, dtype=float)
empirical_risk_open_prices = empirical_risk_open_prices.reindex(index=etf_prices.index, columns=etf_prices.columns)

def _latest_empirical_breach_rate(price_series, open_series, horizon, lookback, alpha):
    clean_prices = pd.to_numeric(price_series, errors='coerce').dropna().sort_index()
    if horizon == 0:
        if open_series is None:
            return np.nan
        holding_frame = pd.DataFrame(
            {
                'Open': pd.to_numeric(open_series, errors='coerce'),
                'Close': pd.to_numeric(price_series, errors='coerce'),
            }
        ).dropna().sort_index()
        if len(holding_frame) < lookback:
            return np.nan
        horizon_returns = holding_frame['Close'].div(holding_frame['Open']).sub(1.0).dropna()
    else:
        if len(clean_prices) <= horizon:
            return np.nan
        horizon_returns = clean_prices.pct_change(periods=horizon, fill_method=None).dropna()

    if len(horizon_returns) < lookback:
        return np.nan

    metric_set = calculate_historical_var_metrics(horizon_returns, window=lookback, alpha=alpha)
    rolling_breach_rate = metric_set['rolling_breach_rate'].dropna()
    breach_series = metric_set['breaches'].dropna()

    if not rolling_breach_rate.empty:
        return rolling_breach_rate.iloc[-1]
    if not breach_series.empty:
        return breach_series.tail(lookback).mean()
    return np.nan

empirical_risk_rows = []
for ticker in etf_prices.columns:
    ticker_row = {'Ticker': ticker}
    open_series = empirical_risk_open_prices[ticker] if ticker in empirical_risk_open_prices.columns else None
    for horizon, horizon_column in zip(empirical_risk_horizons, empirical_risk_horizon_columns):
        ticker_row[horizon_column] = _latest_empirical_breach_rate(
            etf_prices[ticker],
            open_series,
            horizon=horizon,
            lookback=empirical_risk_lookback,
            alpha=empirical_risk_alpha,
        )

    empirical_risk_rows.append(ticker_row)

empirical_risk_summary = pd.DataFrame(
    empirical_risk_rows,
    columns=['Ticker', *empirical_risk_horizon_columns],
)

empirical_risk_header_values = ['Ticker', *empirical_risk_horizon_columns]
empirical_risk_cell_fill_color = '#111827'
empirical_risk_breach_palette = ['#064E3B', '#166534', '#854D0E', '#B45309', '#991B1B']

def _breach_rate_percentile_cell_colors(series):
    values = pd.to_numeric(series, errors='coerce')
    percentiles = values.rank(pct=True)
    colors = []
    for value, percentile in zip(values, percentiles):
        if pd.isna(value) or pd.isna(percentile):
            colors.append(empirical_risk_cell_fill_color)
        elif percentile <= 0.25:
            colors.append(empirical_risk_breach_palette[0])
        elif percentile <= 0.50:
            colors.append(empirical_risk_breach_palette[1])
        elif percentile <= 0.75:
            colors.append(empirical_risk_breach_palette[2])
        elif percentile <= 0.90:
            colors.append(empirical_risk_breach_palette[3])
        else:
            colors.append(empirical_risk_breach_palette[4])
    return colors

def _sort_empirical_risk_display(sort_column):
    if sort_column == 'Ticker':
        return empirical_risk_summary.sort_values('Ticker')
    return empirical_risk_summary.sort_values([sort_column, 'Ticker'], ascending=[False, True])

def _build_empirical_risk_table_payload(sort_column):
    sorted_summary = _sort_empirical_risk_display(sort_column)
    sorted_display = sorted_summary.copy()
    for horizon_column in empirical_risk_horizon_columns:
        sorted_display[horizon_column] = sorted_display[horizon_column].map(
            lambda value: 'N/A' if pd.isna(value) else f'{value:.2%}'
        )
    cell_values = [
        sorted_display['Ticker'].tolist(),
        *[sorted_display[column].tolist() for column in empirical_risk_horizon_columns],
    ]
    cell_fill_colors = [
        [empirical_risk_cell_fill_color] * len(sorted_display),
        *[
            _breach_rate_percentile_cell_colors(sorted_summary[column])
            for column in empirical_risk_horizon_columns
        ],
    ]
    return sorted_display, cell_values, cell_fill_colors

empirical_risk_sort_columns = ['Ticker', *empirical_risk_horizon_columns]
empirical_risk_initial_sort_column = '0D'
empirical_risk_table_payloads = {
    sort_column: _build_empirical_risk_table_payload(sort_column)
    for sort_column in empirical_risk_sort_columns
}
empirical_risk_display, empirical_risk_cell_values, empirical_risk_cell_fill_colors = empirical_risk_table_payloads[empirical_risk_initial_sort_column]
empirical_risk_sort_buttons = [
    dict(
        label=sort_column,
        method='update',
        args=[
            {
                'cells.values': [empirical_risk_table_payloads[sort_column][1]],
                'cells.fill.color': [empirical_risk_table_payloads[sort_column][2]],
            },
            {'title': f'Empirical 95% Breach Rate by Horizon, 50-Day Lookback; 0D = Open-to-Close (sorted by {sort_column})'},
        ],
    )
    for sort_column in empirical_risk_sort_columns
]

empirical_risk_fig = go.Figure(
    data=[
        go.Table(
            header=dict(
                values=empirical_risk_header_values,
                fill_color=['#203040', *(['#0E7490'] * len(empirical_risk_horizon_columns))],
                font=dict(color='white', size=12),
                align='left',
            ),
            cells=dict(
                values=empirical_risk_cell_values,
                fill_color=empirical_risk_cell_fill_colors,
                font=dict(color='#E5E7EB', size=11),
                align='left',
                height=24,
            ),
        )
    ]
)
empirical_risk_fig.update_layout(
    title=f'Empirical 95% Breach Rate by Horizon, 50-Day Lookback; 0D = Open-to-Close (sorted by {empirical_risk_initial_sort_column})',
    template='plotly_dark',
    height=max(460, 24 * len(empirical_risk_display) + 160),
    autosize=True,
    margin=dict(t=100),
    updatemenus=[
        dict(
            buttons=empirical_risk_sort_buttons,
            active=empirical_risk_sort_columns.index(empirical_risk_initial_sort_column),
            direction='down',
            showactive=True,
            x=0,
            xanchor='left',
            y=1.12,
            yanchor='top',
            bgcolor='#111827',
            bordercolor='#374151',
            font=dict(color='white'),
        )
    ],
)
empirical_risk_fig.show(config={'responsive': True})

# Build an options activity table across the current asset universe.
def _sum_chain_numeric_column(chain_frame, column_name):
    if chain_frame is None or chain_frame.empty or column_name not in chain_frame.columns:
        return 0
    return int(pd.to_numeric(chain_frame[column_name], errors='coerce').fillna(0).sum())


def _option_activity_summary_for_ticker(ticker_symbol):
    today = pd.Timestamp.today().normalize()
    total_open_interest = 0
    open_interest_over_200_dte = 0
    open_interest_50_to_200_dte = 0
    open_interest_21_to_50_dte = 0
    open_interest_7_to_21_dte = 0
    open_interest_under_7_dte = 0
    total_option_volume = 0
    option_volume_over_200_dte = 0
    option_volume_50_to_200_dte = 0
    option_volume_21_to_50_dte = 0
    option_volume_7_to_21_dte = 0
    option_volume_under_7_dte = 0

    try:
        ticker_obj = qa_yf.Ticker(ticker_symbol)
        expirations = list(ticker_obj.options or [])
    except Exception as exc:
        return ticker_symbol, {
            'Total Open Interest': 0,
            '>200 DTE Open Interest': 0,
            '50-200 DTE Open Interest': 0,
            '21-50 DTE Open Interest': 0,
            '7-21 DTE Open Interest': 0,
            '<7 DTE Open Interest': 0,
            'Total Option Volume': 0,
            '>200 DTE Option Volume': 0,
            '50-200 DTE Option Volume': 0,
            '21-50 DTE Option Volume': 0,
            '7-21 DTE Option Volume': 0,
            '<7 DTE Option Volume': 0,
        }, str(exc)

    for expiration in expirations:
        try:
            expiration_date = pd.Timestamp(expiration).normalize()
            dte = int((expiration_date - today).days)
            option_chain = ticker_obj.option_chain(expiration)
            expiration_open_interest = (
                _sum_chain_numeric_column(option_chain.calls, 'openInterest')
                + _sum_chain_numeric_column(option_chain.puts, 'openInterest')
            )
            expiration_option_volume = (
                _sum_chain_numeric_column(option_chain.calls, 'volume')
                + _sum_chain_numeric_column(option_chain.puts, 'volume')
            )
        except Exception:
            continue

        total_open_interest += expiration_open_interest
        total_option_volume += expiration_option_volume
        if dte < 7:
            open_interest_under_7_dte += expiration_open_interest
            option_volume_under_7_dte += expiration_option_volume
        elif dte < 21:
            open_interest_7_to_21_dte += expiration_open_interest
            option_volume_7_to_21_dte += expiration_option_volume
        elif dte < 50:
            open_interest_21_to_50_dte += expiration_open_interest
            option_volume_21_to_50_dte += expiration_option_volume
        elif dte <= 200:
            open_interest_50_to_200_dte += expiration_open_interest
            option_volume_50_to_200_dte += expiration_option_volume
        else:
            open_interest_over_200_dte += expiration_open_interest
            option_volume_over_200_dte += expiration_option_volume

    return ticker_symbol, {
        'Total Open Interest': total_open_interest,
        '>200 DTE Open Interest': open_interest_over_200_dte,
        '50-200 DTE Open Interest': open_interest_50_to_200_dte,
        '21-50 DTE Open Interest': open_interest_21_to_50_dte,
        '7-21 DTE Open Interest': open_interest_7_to_21_dte,
        '<7 DTE Open Interest': open_interest_under_7_dte,
        'Total Option Volume': total_option_volume,
        '>200 DTE Option Volume': option_volume_over_200_dte,
        '50-200 DTE Option Volume': option_volume_50_to_200_dte,
        '21-50 DTE Option Volume': option_volume_21_to_50_dte,
        '7-21 DTE Option Volume': option_volume_7_to_21_dte,
        '<7 DTE Option Volume': option_volume_under_7_dte,
    }, None


option_activity_universe = [str(ticker) for ticker in etf_prices.columns]
option_activity_rows = {}
option_activity_errors = {}

with concurrent.futures.ThreadPoolExecutor(max_workers=8) as executor:
    futures = {
        executor.submit(_option_activity_summary_for_ticker, ticker): ticker
        for ticker in option_activity_universe
    }
    for future in concurrent.futures.as_completed(futures):
        ticker_symbol, row, error = future.result()
        option_activity_rows[ticker_symbol] = row
        if error:
            option_activity_errors[ticker_symbol] = error

option_activity_summary = (
    pd.DataFrame.from_dict(option_activity_rows, orient='index')
    .reindex(option_activity_universe)
    .fillna(0)
    .astype(int)
    .sort_values(['Total Open Interest', 'Total Option Volume'], ascending=False)
)
open_interest_summary = option_activity_summary

option_activity_display = option_activity_summary.copy()
option_activity_header_values = ['Ticker', *option_activity_display.columns.tolist()]
option_activity_header_colors = [
    '#203040',
    *(['#0E7490'] * 6),
    *(['#7C3AED'] * 6),
]
option_activity_cell_fill_color = '#111827'
option_activity_open_interest_palette = ['#1F2937', '#164E63', '#0E7490', '#0369A1', '#075985']
option_activity_volume_palette = ['#1F2937', '#3B0764', '#5B21B6', '#7C3AED', '#8B5CF6']

def _metric_percentile_cell_colors(series, palette):
    values = pd.to_numeric(series, errors='coerce').fillna(0)
    percentiles = values.where(values > 0).rank(pct=True)
    colors = []
    for value, percentile in zip(values, percentiles):
        if value <= 0 or pd.isna(percentile):
            colors.append(option_activity_cell_fill_color)
        elif percentile <= 0.25:
            colors.append(palette[0])
        elif percentile <= 0.50:
            colors.append(palette[1])
        elif percentile <= 0.75:
            colors.append(palette[2])
        elif percentile <= 0.90:
            colors.append(palette[3])
        else:
            colors.append(palette[4])
    return colors

def _sort_option_activity_display(sort_column):
    if sort_column == 'Ticker':
        return option_activity_display.sort_index()
    tie_breakers = [column for column in ['Total Open Interest', 'Total Option Volume'] if column != sort_column]
    return option_activity_display.sort_values([sort_column, *tie_breakers], ascending=False)

def _build_option_activity_table_payload(sort_column):
    sorted_display = _sort_option_activity_display(sort_column)
    sorted_formatted = sorted_display.map(lambda value: f'{int(value):,}')
    cell_values = [
        sorted_formatted.index.tolist(),
        *[sorted_formatted[column].tolist() for column in sorted_formatted.columns],
    ]
    cell_fill_colors = [
        [option_activity_cell_fill_color] * len(sorted_formatted.index),
        *[
            _metric_percentile_cell_colors(
                sorted_display[column],
                option_activity_volume_palette if 'Volume' in column else option_activity_open_interest_palette,
            )
            for column in sorted_formatted.columns
        ],
    ]
    return sorted_formatted, cell_values, cell_fill_colors

option_activity_sort_columns = ['Ticker', *option_activity_display.columns.tolist()]
option_activity_initial_sort_column = 'Total Open Interest'
option_activity_table_payloads = {
    sort_column: _build_option_activity_table_payload(sort_column)
    for sort_column in option_activity_sort_columns
}
option_activity_display_formatted, option_activity_cell_values, option_activity_cell_fill_colors = option_activity_table_payloads[option_activity_initial_sort_column]
option_activity_sort_buttons = [
    dict(
        label=sort_column,
        method='update',
        args=[
            {
                'cells.values': [option_activity_table_payloads[sort_column][1]],
                'cells.fill.color': [option_activity_table_payloads[sort_column][2]],
            },
            {'title': f'Options Activity by Asset (sorted by {sort_column})'},
        ],
    )
    for sort_column in option_activity_sort_columns
]
option_activity_fig = go.Figure(
    data=[
        go.Table(
            header=dict(
                values=option_activity_header_values,
                fill_color=option_activity_header_colors,
                font=dict(color='white', size=12),
                align='left',
            ),
            cells=dict(
                values=option_activity_cell_values,
                fill_color=option_activity_cell_fill_colors,
                font=dict(color='#E5E7EB', size=11),
                align='left',
                height=24,
            ),
        )
    ]
)
option_activity_fig.update_layout(
    title=f'Options Activity by Asset (sorted by {option_activity_initial_sort_column})',
    template='plotly_dark',
    height=max(460, 26 * len(option_activity_display_formatted) + 160),
    autosize=True,
    margin=dict(t=100),
    updatemenus=[
        dict(
            buttons=option_activity_sort_buttons,
            active=option_activity_sort_columns.index(option_activity_initial_sort_column),
            direction='down',
            showactive=True,
            x=0,
            xanchor='left',
            y=1.12,
            yanchor='top',
            bgcolor='#111827',
            bordercolor='#374151',
            font=dict(color='white'),
        )
    ],
)
option_activity_fig.show(config={'responsive': True})

open_interest_errors = option_activity_errors
if option_activity_errors:
    print(f'Options activity lookup unavailable for {len(option_activity_errors)} tickers. See option_activity_errors for details.')
